# Early-Exit CNN → Quantised TFLite → C

**Authors:** Ajay Kumar M, Vishnu PS

Flow:
1. Fine-tune a pretrained lightweight CNN on a 10-class classification task (CIFAR-10)
2. Select the exit layers with Algorithm 1 of the paper (hardware-aware exit placement; optionally accuracy-aware
   or with measured layer costs; `software/dynn_placement.py` of the repository)
3. Attach early-exit heads and train them jointly with weighted cross-entropy loss (backbone frozen)
4. Quantise to INT8 TFLite and compare with FP32
5. Compile to C source via Apache TVM

**Why this notebook is different from `model_to_multi_exit_C.ipynb`.** SqueezeNet
has no ImageNet-pretrained checkpoint in `keras.applications`. Part A below
recovers one: the official `torchvision` weights are exported to ONNX and
converted to a Keras (NHWC) model, which is then verified against the PyTorch
reference. Part B is otherwise identical to the base pipeline and consumes the
saved Keras backbone through the same `MODEL_REGISTRY` pattern.

Part A and Part B use **different, incompatible dependency sets**
(PyTorch/ONNX vs. pinned TF 2.12/TVM) and must be run as separate Colab
sessions — restart the runtime between them.


## Part A · PyTorch → ONNX → Keras backbone conversion

Run once per architecture. Produces `squeezenet1_1_backbone.h5`, consumed by
the `squeezenet1_1` entry in `MODEL_REGISTRY` in Part B. Skip this part if
that file already exists on Drive.


### A0 · Install dependencies

In [ ]:
!pip install -q --no-cache-dir "numpy<2.3,>=2.0" torch torchvision --index-url https://download.pytorch.org/whl/cpu
!pip install -q --no-cache-dir --force-reinstall --no-deps torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cpu
!pip install -q tf_keras onnx onnxsim onnx2keras


In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import numpy as np
import torch
import torchvision
import torchvision.models as models
import tensorflow as tf

print("numpy:", np.__version__)
print("torch:", torch.__version__, "| torchvision:", torchvision.__version__)
print("tf:", tf.__version__, "| tf.keras module:", tf.keras.__name__)


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

CONVERTED_DIR = "/content/drive/MyDrive/converted_models"
os.makedirs(CONVERTED_DIR, exist_ok=True)


### A1 · Export SqueezeNet1.1 (ImageNet pretrained) to ONNX

In [ ]:
torch_model = models.squeezenet1_1(weights=models.SqueezeNet1_1_Weights.IMAGENET1K_V1)
torch_model.eval()

dummy_input = torch.randn(1, 3, 64, 64)   # NCHW, matches IMAGE_SIZE spatial dims in Part B
onnx_path = "squeezenet1_1.onnx"
torch.onnx.export(
    torch_model, dummy_input, onnx_path,
    input_names=["input"], output_names=["output"],
    opset_version=13, dynamic_axes=None,
)
print(f"Exported -> {onnx_path}")


### A2 · Simplify and convert to Keras (NHWC)

`onnxsim` folds the exported graph's redundant ops before conversion.
`onnx2keras` then rebuilds it as a Keras model; `change_ordering=True`
transposes NCHW → NHWC to match the rest of the pipeline. `name_policy`
`"renumerate"` produces the sequential `LAYER_nn` names used to index the
exit boundaries and the backbone truncation point below.


In [ ]:
!python -m onnxsim squeezenet1_1.onnx squeezenet1_1_sim.onnx

import onnx
from onnx2keras import onnx_to_keras
import tf_keras.src.saving.serialization_lib as _ser_lib
_ser_lib.in_safe_mode = lambda: False

onnx_model = onnx.load("squeezenet1_1_sim.onnx")
keras_model_nhwc = onnx_to_keras(
    onnx_model, ["input"],
    name_policy="renumerate",
    change_ordering=True,
    verbose=False,
)
keras_model_nhwc.summary()


### A3 · Validate against the PyTorch model

Cosine similarity on raw logits for one random input.


In [ ]:
np.random.seed(0)
test_input_np = np.random.rand(1, 3, 64, 64).astype(np.float32)   # NCHW

with torch.no_grad():
    torch_out = torch_model(torch.from_numpy(test_input_np)).numpy().squeeze()

keras_input_nhwc = np.transpose(test_input_np, (0, 2, 3, 1))      # NCHW -> NHWC
keras_out_nhwc = keras_model_nhwc.predict(keras_input_nhwc, verbose=0)
if isinstance(keras_out_nhwc, dict):
    keras_out_nhwc = list(keras_out_nhwc.values())[0]
keras_out_nhwc = np.array(keras_out_nhwc).squeeze()

max_abs_diff = np.max(np.abs(torch_out - keras_out_nhwc))
cos_sim = np.dot(torch_out, keras_out_nhwc) / (
    np.linalg.norm(torch_out) * np.linalg.norm(keras_out_nhwc))
print(f"Max abs diff: {max_abs_diff:.6f}   Cosine sim: {cos_sim:.6f}")
print("Top-5 Keras:  ", np.argsort(keras_out_nhwc)[-5:][::-1])
print("Top-5 PyTorch:", np.argsort(torch_out)[-5:][::-1])
assert cos_sim > 0.999, "Conversion mismatch — do not proceed to Part B on this backbone."


### A4 · Extract the backbone and save

`LAYER_60` is the last Fire-module concatenation — the final spatial feature
map before SqueezeNet's classifier conv — i.e. the ONNX-converted equivalent
of `include_top=False` for the other registry entries. Re-derive this index
with `keras_model_nhwc.summary()` above if `onnx2keras`'s layer numbering
ever changes between versions.


In [ ]:
SQUEEZENET_BACKBONE_PATH = os.path.join(CONVERTED_DIR, "squeezenet1_1_backbone.h5")

squeezenet_backbone = tf.keras.Model(
    inputs=keras_model_nhwc.input,
    outputs=keras_model_nhwc.get_layer("LAYER_60").output,
    name="squeezenet1_1_backbone",
)
squeezenet_backbone.summary()
squeezenet_backbone.save(SQUEEZENET_BACKBONE_PATH)
print(f"Saved -> {SQUEEZENET_BACKBONE_PATH}")

reloaded = tf.keras.models.load_model(SQUEEZENET_BACKBONE_PATH)
assert reloaded.output_shape == (None, 3, 3, 512), reloaded.output_shape
print("Reload check OK. Output shape:", reloaded.output_shape)


## Part B · Multi-exit model

Identical structure to `model_to_multi_exit_C.ipynb`. **Restart the runtime**
before this section — Part A's PyTorch/ONNX stack and this section's
TF 2.12 / TVM stack are not compatible in one process.


### 0 · Install dependencies

In [ ]:
!pip install -q tensorflow==2.12.0 keras==2.12.0 tflite==2.1.0 numpy==1.23.5
!pip install -q apache-tvm==0.14.dev273


In [ ]:
!pip install "https://files.pythonhosted.org/packages/9c/b2/adb56267ba6b6f05204407a7745e0c188e39589b7e33f490d4aa406d84e8/apache_tvm-0.14.dev273-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl"

Restart session to load the dependencies. Some errors may be seen when installing the TVM version. They can be ignored.

## 1 · Configuration

Edit **`MODEL_KEY`** to switch architectures.


In [ ]:
import os, pathlib, shutil, urllib.request, tarfile, pickle
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tvm, tvm.relay as relay
from tvm.relay.backend import Executor, Runtime
from tvm.micro import export_model_library_format

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# Dataset
IMAGE_SIZE      = (64, 64, 3)
BATCH_SIZE      = 32
NUM_CLASSES     = 10
CIFAR_CLASSES   = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']
UART_SAMPLES    = 50                    # samples per class for UART test export
CACHE_DIR       = "/content/drive/MyDrive/cached_datasets"

# Training
LEARNING_RATE   = 5e-4
EPOCHS_FINETUNE = 10
EPOCHS_EE       = 10
DROPOUT_RATE    = 0.1

# Early-exit
EE_LOSS_WEIGHTS = [0.2, 0.4, 0.7, 1.0]   # weight per exit (last = final head)

# Exit placement: Algorithm 1 of the paper (software/dynn_placement.py)
AUTO_SELECT_EXITS = True        # False = use exit_layers from the registry
PLACEMENT_MODE    = "depth"     # "depth":    depth-based selection (the networks of the paper)
                                # "accuracy": accuracy-aware option (Algorithm 1, line 17): an exit head is trained at
                                #             every candidate layer and evaluated on the validation split
ALPHA_ACC         = 0.25        # accelerator cost factor on the BB core (1.0 = no accelerator)
MEASURED_COSTS    = None        # or the measured cost of every backbone layer on the target (Algorithm 1, line 1),
                                # e.g. "software/measured_costs/SqueezeNet1_1_DEA.json"; replaces ALPHA_ACC
N_EXITS_TARGET    = 3           # N_max; the search reduces N if no configuration is feasible
N_EXITS_MIN       = 1
MIN_FIRST_DEPTH   = 10.0        # D_min (% of the backbone cost)
LAST_DEPTH_RANGE  = (55.0, 90.0)  # [D_L,min, D_L,max]
TARGET_LAST_DEPTH = 70.0        # D_tar
LAMBDA_BAL        = 10.0        # lambda, weight of the segment-balance term
ACC_MARGIN        = 2.0         # Delta of the accuracy-aware option (accuracy points below the no-exit accuracy)
# Candidate exits: layers with a 4-D output; only the architectural boundaries below if there are >= 2 * N_max
BOUNDARY_KEYWORDS       = ["Add", "project/BatchNorm", "_pool", "_concat", "block_pool", "_out", "pool_conv"]
BOUNDARY_CASE_SENSITIVE = False
BOUNDARY_LAYER_TYPES    = (tf.keras.layers.Concatenate, tf.keras.layers.MaxPooling2D,
                           tf.keras.layers.AveragePooling2D)   # SqueezeNet: Fire-module concatenations
BOUNDARY_DEDUP          = False
REPO_REF          = "main"      # branch or tag of the repository from which dynn_placement.py is loaded

# TVM
TVM_OPT_LEVEL = 3
QUANTIZE_FOR_TVM = True                   # Set to True for quantized, False for unquantized C code

# Fine-tune control
FINETUNE_FROM_SCRATCH = True             # True  = train backbone from imagenet
                                         # False = load saved weights from Drive
FINETUNED_WEIGHTS_DIR = "/content/drive/MyDrive/finetuned_weights"
CONVERTED_DIR         = "/content/drive/MyDrive/converted_models"   # SqueezeNet backbone of Part A


In [ ]:
# Algorithm 1 (dynn_placement.py) and the measured layer costs are in the folder software/ of the repository.
import sys, json, hashlib, urllib.request
REPO_URL = f"https://raw.githubusercontent.com/ajaymahendersingh/DyNN_RV_SoC/{REPO_REF}/"

def repo_file(relpath):
    """Path of a file of the repository: the local copy if the notebook runs in a clone, else downloaded once."""
    for p in (pathlib.Path(relpath), pathlib.Path(relpath).relative_to("software") if relpath.startswith("software/")
              else None, pathlib.Path(pathlib.Path(relpath).name)):
        if p is not None and p.exists():
            return str(p)
    urllib.request.urlretrieve(REPO_URL + relpath, pathlib.Path(relpath).name)
    return pathlib.Path(relpath).name

sys.path.insert(0, os.path.dirname(os.path.abspath(repo_file("software/dynn_placement.py"))))
import dynn_placement as DP

## 2 · Load data

In [ ]:
MODEL_REGISTRY = {

    "mobilenet_v1": {
        "name": "MobileNetV1",
        "build_fn": lambda: tf.keras.applications.MobileNet(
            input_shape=IMAGE_SIZE, alpha=0.25, weights="imagenet", include_top=False
        ),
        "exit_layers": ["conv_pw_2_relu", "conv_pw_4_relu", "conv_pw_6_relu"],
        "head_offset": -5,
    },

    "mobilenet_v2": {
        "name": "MobileNetV2",
        "build_fn": lambda: tf.keras.applications.MobileNetV2(
            input_shape=IMAGE_SIZE, alpha=0.35, weights="imagenet", include_top=False
        ),
        "exit_layers": ["block_3_expand_relu", "block_6_expand_relu",
                        "block_10_expand_relu"],
        "head_offset": -1,
    },

    "mobilenet_v3_small": {
        "name": "MobileNetV3Small",
        "build_fn": lambda: tf.keras.applications.MobileNetV3Small(
            input_shape=IMAGE_SIZE, alpha=0.75, weights="imagenet",
            include_top=False, include_preprocessing=False,
        ),
        "exit_layers": ["expanded_conv_3/project",
                        "expanded_conv_6/project",
                        "expanded_conv_8/project"],
        "head_offset": -1,
    },

    "efficientnet_b0": {
        "name": "EfficientNetB0",
        "build_fn": lambda: tf.keras.applications.EfficientNetB0(
            input_shape=IMAGE_SIZE, weights="imagenet", include_top=False,
        ),
        "exit_layers": ["block2a_expand_activation", "block3a_expand_activation",
                        "block5a_expand_activation"],
        "head_offset": -1,
        "expects_0_255": True,    # flag: this model preprocesses internally
    },

    "resnet50": {
        "name": "ResNet50",
        "build_fn": lambda: tf.keras.applications.ResNet50(
            input_shape=IMAGE_SIZE, weights="imagenet", include_top=False,
        ),
        "exit_layers": ["conv2_block3_out", "conv3_block4_out", "conv4_block6_out"],
        "head_offset": -1,
    },

    "densenet121": {
        "name": "DenseNet121",
        "build_fn": lambda: tf.keras.applications.DenseNet121(
            input_shape=IMAGE_SIZE, weights="imagenet", include_top=False,
        ),
        "exit_layers": ["pool2_conv", "pool3_conv", "pool4_conv"],
        "head_offset": -1,
    },

    "vgg16": {
        "name": "VGG16",
        "build_fn": lambda: tf.keras.applications.VGG16(
            input_shape=IMAGE_SIZE, weights="imagenet", include_top=False,
        ),
        "exit_layers": ["block2_pool", "block3_pool", "block4_pool"],
        "head_offset": -1,
    },

    "nasnet_mobile": {
        "name": "NASNetMobile",
        "build_fn": lambda: tf.keras.applications.NASNetMobile(
            input_shape=IMAGE_SIZE, weights="imagenet", include_top=False,
        ),
        # NASNet cells: stem → 4 Normal → Reduction → 4 Normal → Reduction → 4 Normal
        # Run backbone.summary() to verify layer names for your TF version.
        "exit_layers": ["normal_concat_3", "normal_concat_6",
                        "normal_concat_9"],
        "head_offset": -1,
        "tvm_custom_pipeline": False,  # NASNet's branchy graph makes the
        "tvm_opt_level": 1,            # custom fold-before-QNN pipeline hang.
                                       # Standard pipeline compiles fine.
    },

    "efficientnet_v2_b0": {
        "name": "EfficientNetV2B0",
        "build_fn": lambda: tf.keras.applications.EfficientNetV2B0(
            input_shape=IMAGE_SIZE, weights="imagenet",
            include_top=False, include_preprocessing=False,
        ),
        # EfficientNetV2B0 stages: block1 → block2 → ... → block6
        # Exit points at ~25/50/75% depth across stages.
        "exit_layers": ["block2b_add", "block4a_expand_activation",
                        "block5e_add"],
        "head_offset": -1,
    },

    "squeezenet1_1": {
        "name": "SqueezeNet1_1",
        "build_fn": lambda: tf.keras.models.load_model(
            os.path.join(CONVERTED_DIR, "squeezenet1_1_backbone.h5")   # written by Part A
        ),
        "exit_layers": ["LAYER_16", "LAYER_31", "LAYER_53"],  # Fire4/6/8 concat boundaries
        "head_offset": -1,
    },

}


In [ ]:
# ━━━━━━━━━  CHOOSE YOUR MODEL HERE  ━━━━━━━━━━━━━━━━━━━━━━━━
MODEL_KEY = "squeezenet1_1"
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

MODEL_CFG  = MODEL_REGISTRY[MODEL_KEY]
# One run directory per model and placement setting, so that runs with different exits do not overwrite each other.
PLACEMENT_TAG = ("registry" if not AUTO_SELECT_EXITS else
                 ("measured" if MEASURED_COSTS else f"alpha{ALPHA_ACC:g}")
                 + ("_accuracy" if PLACEMENT_MODE == "accuracy" else ""))
RUN_TAG    = f"{MODEL_CFG['name']}_cifar{NUM_CLASSES}_{IMAGE_SIZE[0]}_{PLACEMENT_TAG}"
OUTPUT_DIR = pathlib.Path(f"./output_{RUN_TAG}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Model: {MODEL_CFG['name']}   Output: {OUTPUT_DIR}")

In [ ]:
# SqueezeNet-specific sanity check: this backbone's stage boundaries are Fire-module
# Concatenate layers, not the Add/pool boundaries the other registry entries use.
# Confirms BOUNDARY_KEYWORDS + isinstance(Concatenate) in the exit-placement search
# below actually finds candidates on this architecture before it runs.
test_backbone = MODEL_CFG["build_fn"]()
concat_layers = [l.name for l in test_backbone.layers if type(l).__name__ == "Concatenate"]
print(f"Found {len(concat_layers)} Concatenate layers: {concat_layers}")
del test_backbone


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# ── Try loading cached arrays from Drive ─────────────────
cache_path = os.path.join(CACHE_DIR, f"cifar{NUM_CLASSES}_{IMAGE_SIZE[0]}x{IMAGE_SIZE[1]}.npz")
os.makedirs(CACHE_DIR, exist_ok=True)

if os.path.exists(cache_path):
    print(f"Loading cached dataset from {cache_path} ...")
    data = np.load(cache_path)
    x_train_all, y_train_all = data["x_train"], data["y_train"]
    x_test_all, y_test_all   = data["x_test"],  data["y_test"]
    print(f"  train: {len(x_train_all)},  test holdout: {len(x_test_all)}")

else:
    print("Cache not found — building dataset from scratch ...")
    (x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = tf.keras.datasets.cifar10.load_data()

    # Rescale to [0,1]
    x_train_all = x_train_raw.astype("float32") / 255.0
    y_train_all = y_train_raw.squeeze()
    x_test_all  = x_test_raw.astype("float32") / 255.0
    y_test_all  = y_test_raw.squeeze()

    # Resize 32x32 → IMAGE_SIZE
    x_train_all = tf.image.resize(x_train_all, IMAGE_SIZE[:2]).numpy()
    x_test_all  = tf.image.resize(x_test_all, IMAGE_SIZE[:2]).numpy()

    # One-hot encode
    y_train_all = tf.keras.utils.to_categorical(y_train_all, NUM_CLASSES)
    y_test_all  = tf.keras.utils.to_categorical(y_test_all, NUM_CLASSES)

    # Save to Drive for next time
    np.savez_compressed(cache_path,
                        x_train=x_train_all, y_train=y_train_all,
                        x_test=x_test_all,   y_test=y_test_all)
    print(f"Cached processed arrays → {cache_path}")

# Shuffle training data and split 80/20 → train/val
idx = np.random.permutation(len(x_train_all))
x_train_all, y_train_all = x_train_all[idx], y_train_all[idx]
split = int(0.8 * len(x_train_all))

train_ds = tf.data.Dataset.from_tensor_slices(
    (x_train_all[:split], y_train_all[:split])
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = tf.data.Dataset.from_tensor_slices(
    (x_train_all[split:], y_train_all[split:])
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
full_ds = tf.data.Dataset.from_tensor_slices(
    (x_train_all, y_train_all)
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Validation split (accuracy-aware exit placement) and test set
x_val, y_val = x_train_all[split:], y_train_all[split:]
test_images = x_test_all
test_labels = y_test_all

print(f"CIFAR-10 — {NUM_CLASSES} classes: {CIFAR_CLASSES}")
print(f"train: {len(train_ds)} batches,  val: {len(val_ds)} batches,  full: {len(full_ds)} batches")
print(f"Test set: {len(test_images)} images (from original CIFAR-10 test set)")


## 3 · Fine-tune the backbone

In [ ]:
pretrained = MODEL_CFG["build_fn"]()

# ── Build the backbone model ─────────────────────────────
backbone_model = tf.keras.Model(
    inputs=pretrained.inputs,
    outputs=pretrained.layers[MODEL_CFG["head_offset"]].output,
)

layers = [tf.keras.layers.InputLayer(input_shape=IMAGE_SIZE)]
if MODEL_CFG.get("expects_0_255"):
    layers.append(tf.keras.layers.Rescaling(255.0))

layers += [
    backbone_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(DROPOUT_RATE),
    tf.keras.layers.Dense(NUM_CLASSES, activation="softmax"),
]
base_model = tf.keras.Sequential(layers)

base_model.compile(
    optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
    loss="categorical_crossentropy", metrics=["accuracy"],
)

weights_path = os.path.join(FINETUNED_WEIGHTS_DIR,
                            f"{MODEL_CFG['name']}_cifar{NUM_CLASSES}_finetuned.weights.h5")

if FINETUNE_FROM_SCRATCH:
    print(f"Fine-tuning {MODEL_CFG['name']} from ImageNet weights ...")
    history = base_model.fit(train_ds, validation_data=val_ds,
                             epochs=EPOCHS_FINETUNE, verbose=2)
else:
    print(f"Loading saved weights from {weights_path} ...")
    base_model.load_weights(weights_path)
    history = None
    print("Loaded. Skipping fine-tune.")


In [ ]:
if FINETUNE_FROM_SCRATCH:
    os.makedirs(FINETUNED_WEIGHTS_DIR, exist_ok=True)
    base_model.save_weights(weights_path)
    print(f"Saved fine-tuned weights → {weights_path}")


In [ ]:
def plot_history(hist, title="", save_path=None):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
    for ax, m in zip([a1, a2], ["loss", "accuracy"]):
        train_key = m
        val_key   = f"val_{m}"
        if train_key not in hist.history:
            candidates = [k for k in hist.history if k.endswith(f"_{m}") and not k.startswith("val_")]
            if candidates:
                train_key = candidates[-1]
                val_key   = f"val_{train_key}"
        ax.plot(hist.history[train_key], label="train")
        ax.plot(hist.history[val_key],   label="val")
        ax.set(xlabel="Epoch", ylabel=m.title(), title=f"{title}{m.title()}")
        ax.legend()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Plot saved to {save_path}")
    plt.show()

if history is not None:
    plot_history(history, f"{MODEL_CFG['name']} base — ", save_path=OUTPUT_DIR / f"{MODEL_CFG['name']}_base_training_history.png")
else:
    print("Skipped base training plot — loaded weights from Drive.")


In [ ]:
base_model.summary()

In [ ]:
backbone = None
for layer in base_model.layers:
    if isinstance(layer, tf.keras.Model):
        backbone = layer
        break

if backbone:
    backbone.summary()
else:
    print("Could not find the backbone (Functional model) within base_model.")

In [ ]:
def make_exit_head(features, name, exit_idx=1):

    if exit_idx == 2 or exit_idx == 3:
        x = tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu",
                                   name=f"conv1_{name}")(features)
        x = tf.keras.layers.Conv2D(16, 3, padding="same", activation="relu",
                                   name=f"conv2_{name}")(x)
        x = tf.keras.layers.GlobalAveragePooling2D(name=f"gap_{name}")(x)
        x = tf.keras.layers.Dense(128, activation="relu", name=f"dense_mid_1_{name}")(x)
        x = tf.keras.layers.Dense(64, activation="relu", name=f"dense_mid_2_{name}")(x)

    elif exit_idx == 1:
        x = tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu",
                                   name=f"conv1_{name}")(features)
        x = tf.keras.layers.GlobalAveragePooling2D(name=f"gap_{name}")(x)
        x = tf.keras.layers.Dense(128, activation="relu", name=f"dense_mid_1_{name}")(x)
        x = tf.keras.layers.Dense(64, activation="relu", name=f"dense_mid_2_{name}")(x)

    x = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", name=name)(x)
    return x


## Exit placement (Algorithm 1)

**Cost model.** Only the BB core reaches the dot-product accelerator. Backbone MACs that are accelerator-eligible
(1x1 Conv2D, Dense) are scaled by `ALPHA_ACC`; every other backbone MAC and *all* exit-head MACs are charged at full
software cost, because the EE core runs the heads without the accelerator. If the network has been profiled on the
target, `MEASURED_COSTS` replaces this model by the measured cost of every layer (line 1 of Algorithm 1; the head
cost is then the measured cycles per MAC of the EE core times the head MACs). Measured costs of the five networks of
the paper on DE+A are in `software/measured_costs/`.

**Depth-based selection.** The first exit must lie past `MIN_FIRST_DEPTH` % of the backbone cost and the last exit in
`LAST_DEPTH_RANGE`; every exit head must cost less than the backbone segment that follows it (for the last exit, the
rest of the backbone), so that the EE core finishes before the BB core reaches the next exit. Among the feasible
configurations, the one whose last exit is closest to `TARGET_LAST_DEPTH` and whose segments are best balanced
(`LAMBDA_BAL`) is selected. If no configuration with `N_EXITS_TARGET` exits is feasible, N is reduced down to
`N_EXITS_MIN`. With the default settings, this gives the exits of the networks in the paper.

**Accuracy-aware option** (`PLACEMENT_MODE = "accuracy"`, line 17 of Algorithm 1): see the next cell.

In [ ]:
if AUTO_SELECT_EXITS:
    # One row per backbone layer: name, output shape, accelerator-eligible and software MACs, boundary flag
    table = DP.layer_table(backbone, BOUNDARY_KEYWORDS, BOUNDARY_CASE_SENSITIVE, BOUNDARY_LAYER_TYPES)
    if MEASURED_COSTS:
        layer_cost, head_per_mac = DP.load_measured_costs(repo_file(MEASURED_COSTS), table)
        head_cost = lambda hwc, k: head_per_mac * DP.probe_head_cost(make_exit_head, hwc, k)
        print(f"Measured layer costs: {MEASURED_COSTS} ({head_per_mac:.2f} cycles per head MAC)")
    else:
        layer_cost = None
        head_cost = lambda hwc, k: DP.probe_head_cost(make_exit_head, hwc, k)   # MACs of make_exit_head
    PLACEMENT_N, PLACEMENT_CONFIGS, cand, cum = DP.place(
        table, head_cost, alpha=ALPHA_ACC, layer_cost=layer_cost, d_min=MIN_FIRST_DEPTH,
        last_range=LAST_DEPTH_RANGE, d_tar=TARGET_LAST_DEPTH, lam=LAMBDA_BAL,
        n_max=N_EXITS_TARGET, n_min=N_EXITS_MIN, dedup=BOUNDARY_DEDUP)
    print(f"Backbone: {len(table['name'])} layers, total cost {cum[-1]:,.0f}"
          + ("" if MEASURED_COSTS else f" (alpha_acc={ALPHA_ACC})") + f"; {len(cand)} candidate exits (* boundary)")
    for i in cand:
        print(f"  [{i:3d}] {table['name'][i]:<45s} shape={table['hwc'][i]}  "
              f"depth={100 * cum[i] / cum[-1]:5.1f}%{' *' if table['boundary'][i] else ''}")
    if PLACEMENT_N is None:
        print(f"\nNo feasible placement for N in [{N_EXITS_MIN}, {N_EXITS_TARGET}]: "
              f"using exit_layers from the registry: {MODEL_CFG['exit_layers']}")
    else:
        print(f"\nN = {PLACEMENT_N}: {len(PLACEMENT_CONFIGS)} feasible configuration(s), best by (phi, psi):")
        DP.print_options(PLACEMENT_CONFIGS)
        MODEL_CFG["exit_layers"] = PLACEMENT_CONFIGS[0]["exits"]
        print(f"\nDepth-based selection: {MODEL_CFG['exit_layers']}")
else:
    PLACEMENT_N, PLACEMENT_CONFIGS = None, []
    print(f"AUTO_SELECT_EXITS = False: using exit_layers from the registry: {MODEL_CFG['exit_layers']}")
MODEL_CFG["n_exits_used"] = len(MODEL_CFG["exit_layers"])

### Accuracy-aware option

Runs only with `PLACEMENT_MODE = "accuracy"`. An exit head is trained at every layer that occurs in a feasible
configuration (the exit-1 head where the layer is a first exit, the deeper head where it is a later exit), with the
backbone frozen and the optimizer, learning rate and epochs of the exit training below. Because the backbone is
frozen, the heads are independent, so they are trained together in one model. All heads are then quantized to INT8
together with the network, as deployed, and early exiting is simulated on the validation split for the nine threshold
profiles of the paper: an image leaves at the first exit whose confidence reaches the threshold and costs the backbone
up to that exit plus the head; otherwise it costs the whole backbone. The configuration with the lowest mean cost
per image at an accuracy `ACC_MARGIN` points below the accuracy without exits (interpolated between the profiles) is
selected. This takes longer than the depth-based selection (one training of all candidate heads).

In [ ]:
if AUTO_SELECT_EXITS and PLACEMENT_MODE == "accuracy" and PLACEMENT_N is not None:
    pairs = DP.head_pairs(PLACEMENT_CONFIGS)                     # (layer, head type) to train
    taps_at = sorted({n for n, _ in pairs}, key=table["name"].index)
    print(f"{len(PLACEMENT_CONFIGS)} configurations: {len(pairs)} exit heads at {len(taps_at)} layers")

    # One model: frozen backbone, final classifier of the fine-tuned model, one head per (layer, head type)
    backbone.trainable = False
    tail = base_model.layers[base_model.layers.index(backbone) + 1:]
    tail_trainable = [l.trainable for l in tail]
    for l in tail:
        l.trainable = False
    inp = tf.keras.Input(shape=IMAGE_SIZE, name="image")
    x = tf.keras.layers.Rescaling(255.0, name="rescale_to_255")(inp) if MODEL_CFG.get("expects_0_255") else inp
    taps = tf.keras.Model(backbone.input, [backbone.get_layer(n).output for n in taps_at] + [backbone.output],
                          name="candidate_taps")(x)
    head_name = {p: f"cand_{taps_at.index(p[0])}_{p[1]}" for p in pairs}
    heads = [make_exit_head(taps[taps_at.index(p[0])], head_name[p], exit_idx=p[1]) for p in pairs]
    final = taps[-1]
    for l in tail:
        final = l(final)
    cand_model = tf.keras.Model(inp, [final] + heads, name="candidate_heads")
    final_name = cand_model.output_names[0]
    cand_model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
                       loss={n: (None if n == final_name else "categorical_crossentropy")
                             for n in cand_model.output_names})
    cand_model.fit(train_ds, epochs=EPOCHS_EE, verbose=2)
    for l, t in zip(tail, tail_trainable):
        l.trainable = t

    # INT8 as deployed (converter settings of the quantisation step below); outputs matched by name
    def calibration_images():
        for batch, _ in full_ds.take(100):
            yield [batch]
    conv = tf.lite.TFLiteConverter.from_keras_model(cand_model)
    conv.optimizations             = [tf.lite.Optimize.DEFAULT]
    conv.representative_dataset    = calibration_images
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    conv.inference_input_type      = tf.float32
    conv.inference_output_type     = tf.float32
    out = DP.run_tflite(conv.convert(), x_val)
    y_v = np.argmax(y_val, axis=1)
    head_outputs = {p: (out[head_name[p]].max(1), out[head_name[p]].argmax(1)) for p in pairs}

    sel = DP.select_accuracy_aware(PLACEMENT_CONFIGS, cum, cum[-1], head_outputs, out[final_name].argmax(1), y_v,
                                   ACC_MARGIN)
    print(f"\nValidation split ({len(y_v)} images), INT8; C_Delta in % of the backbone cost:")
    for c in sorted(PLACEMENT_CONFIGS, key=lambda c: c["c_delta"])[:5] + [PLACEMENT_CONFIGS[0]]:
        acc = [float((head_outputs[(n, 1 if k == 0 else 2)][1] == y_v).mean() * 100) for k, n in enumerate(c["exits"])]
        tag = "  <- depth-based" if c is PLACEMENT_CONFIGS[0] else ("  <- selected" if c is sel else "")
        print(f"  C_Delta {100 * c['c_delta'] / cum[-1]:5.1f}%  {c['exits']}  head acc {[round(a, 1) for a in acc]}  "
              f"best-EE {c['best']['profile']} ({c['best']['acc']:.1f}%){tag}")
    MODEL_CFG["exit_layers"] = sel["exits"]
    MODEL_CFG["n_exits_used"] = len(sel["exits"])
    print(f"\nAccuracy-aware selection: {sel['exits']}")

In [ ]:
# Record of the exit selection, next to the outputs of this run
if AUTO_SELECT_EXITS and PLACEMENT_N is not None:
    rec = dict(model=MODEL_CFG["name"], mode=PLACEMENT_MODE, alpha_acc=ALPHA_ACC, measured_costs=MEASURED_COSTS,
               d_min=MIN_FIRST_DEPTH, last_range=LAST_DEPTH_RANGE, d_tar=TARGET_LAST_DEPTH, lam=LAMBDA_BAL,
               acc_margin=ACC_MARGIN, n_exits=PLACEMENT_N, n_configurations=len(PLACEMENT_CONFIGS),
               depth_based=PLACEMENT_CONFIGS[0]["exits"], selected=MODEL_CFG["exit_layers"],
               depths=next(c["depths"] for c in PLACEMENT_CONFIGS if c["exits"] == MODEL_CFG["exit_layers"]))
    (OUTPUT_DIR / "exit_placement.json").write_text(json.dumps(rec, indent=1))
    print(f"Exit placement -> {OUTPUT_DIR / 'exit_placement.json'}")

## 4 · Attach early-exit heads

The backbone is frozen; only exit heads are trained jointly.


In [ ]:
def build_multi_exit_model(base_model, cfg):
    backbone = None
    for layer in base_model.layers:
        if isinstance(layer, tf.keras.Model):
            backbone = layer
            break
    backbone.trainable = False

    inp = tf.keras.Input(shape=IMAGE_SIZE)
    if cfg.get("expects_0_255"):
        x = tf.keras.layers.Rescaling(255.0, name="rescale_to_255")(inp)
    else:
        x = inp

    backbone_out = backbone(x)

    exits = []
    for i, layer_name in enumerate(cfg["exit_layers"], start=1):
        feat = backbone.get_layer(layer_name).output
        exits.append(make_exit_head(feat, f"early_exit_{i}", exit_idx=i))

    x_final = backbone.output
    backbone_idx = base_model.layers.index(backbone)
    for layer in base_model.layers[backbone_idx + 1:]:
        x_final = layer(x_final)
    exits.append(x_final)

    return tf.keras.Model(inputs=backbone.input, outputs=exits,
                          name=f"{cfg['name']}_multi_exit")


multi_exit_model = build_multi_exit_model(base_model, MODEL_CFG)
multi_exit_model.summary()
print("Outputs:", multi_exit_model.output_names)


In [ ]:
output_names = multi_exit_model.output_names
losses  = {n: "categorical_crossentropy" for n in output_names}
weights = dict(zip(output_names, EE_LOSS_WEIGHTS))

# The saved model is specific to the exit layers (a model with other exits is never loaded by mistake)
exit_tag = hashlib.sha1("|".join(MODEL_CFG["exit_layers"]).encode()).hexdigest()[:8]
model_save_path = (pathlib.Path(FINETUNED_WEIGHTS_DIR).parent / OUTPUT_DIR.name
                   / f"{MODEL_CFG['name']}_multi_exit_{exit_tag}.h5")
print(f"Exit layers: {MODEL_CFG['exit_layers']}")
print(f"Attempting to load/save model at: {model_save_path}")

if model_save_path.exists():
    print(f"Loading multi-exit model from {model_save_path}...")
    multi_exit_model = tf.keras.models.load_model(str(model_save_path))
    ee_history = None
    print("Model loaded.")
else:
    print("Multi-exit model not found. Training...")
    model_save_path.parent.mkdir(parents=True, exist_ok=True)
    multi_exit_model.compile(
        optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
        loss=losses, loss_weights=weights, metrics=["accuracy"],
    )
    ee_history = multi_exit_model.fit(train_ds, validation_data=val_ds,
                                      epochs=EPOCHS_EE, verbose=2)
    multi_exit_model.save(str(model_save_path))
    print(f"Multi-exit model saved to {model_save_path}.")

In [ ]:
if ee_history is not None:
    plot_history(ee_history, f"{MODEL_CFG['name']} multi-exit — ", save_path=OUTPUT_DIR / f"{MODEL_CFG['name']}_multi_exit_training_history.png")
else:
    print("Skipped multi-exit training plot — model loaded from Drive.")

## 5 · INT8 quantisation

In [ ]:
def representative_dataset():
    for batch, _ in full_ds.take(100):
        yield [batch]

# Unquantized TFLite conversion
unquantized_converter = tf.lite.TFLiteConverter.from_keras_model(multi_exit_model)
unquantized_tfl_path = OUTPUT_DIR / f"{MODEL_CFG['name']}_multi_exit_fp32.tflite"
unquantized_model = unquantized_converter.convert()
unquantized_tfl_path.write_bytes(unquantized_model)
print(f"FP32 TFLite → {unquantized_tfl_path} ({len(unquantized_model):,} bytes)")

quantized_tfl_path = None
if QUANTIZE_FOR_TVM:
    # Quantized TFLite conversion
    converter = tf.lite.TFLiteConverter.from_keras_model(multi_exit_model)
    converter.optimizations             = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset    = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type      = tf.float32
    converter.inference_output_type     = tf.float32

    quantized = converter.convert()
    quantized_tfl_path = OUTPUT_DIR / f"{MODEL_CFG['name']}_multi_exit_int8.tflite"
    quantized_tfl_path.write_bytes(quantized)
    print(f"Quantized TFLite → {quantized_tfl_path} ({len(quantized):,} bytes)")
else:
    print("Skipping INT8 quantization as QUANTIZE_FOR_TVM is False.")

tflite_to_compile_path = quantized_tfl_path if QUANTIZE_FOR_TVM else unquantized_tfl_path
print(f"Model selected for TVM compilation: {tflite_to_compile_path}")


### FP32 vs. INT8 on the test set

Accuracy of every exit of the FP32 and INT8 models on the test images, their top-1 agreement and the mean change of
the confidence (maximum softmax probability) that the thresholds are compared with. TFLite orders the outputs
alphabetically, so they are matched by name, not by position. Then early exiting with the INT8 confidences for the
nine threshold profiles: accuracy, share of images at each exit and, for exits chosen by Algorithm 1, the mean cost
per image in its cost units. The best-EE profile is the fastest one within 2 points of the accuracy without exits.

In [ ]:
y_t = np.argmax(test_labels, axis=1)
res = {"FP32": DP.run_tflite(unquantized_tfl_path, test_images)}
if quantized_tfl_path is not None:
    res["INT8"] = DP.run_tflite(quantized_tfl_path, test_images)
outs = multi_exit_model.output_names                 # early_exit_1 ... early_exit_N, then the final classifier
print(f"{'output':<16s}" + "".join(f"{k + ' acc':>11s}" for k in res)
      + ("   top-1 agree   mean |d conf|" if "INT8" in res else ""))
for n in outs:
    line = f"{n:<16s}" + "".join(f"{(r[n].argmax(1) == y_t).mean() * 100:10.2f}%" for r in res.values())
    if "INT8" in res:
        a, b = res["FP32"][n], res["INT8"][n]
        line += f"   {(a.argmax(1) == b.argmax(1)).mean() * 100:10.2f}%   {np.abs(a.max(1) - b.max(1)).mean():12.4f}"
    print(line)

if "INT8" in res and len(outs) > 1:
    q8 = res["INT8"]; ex = outs[:-1]
    cfg = next((c for c in PLACEMENT_CONFIGS if c["exits"] == MODEL_CFG["exit_layers"]), None)
    cost = [cum[i] + h for i, h in zip(cfg["idx"], cfg["head_costs"])] if cfg else [0.0] * len(ex)
    rows = DP.simulate([q8[n].max(1) for n in ex], [q8[n].argmax(1) for n in ex], q8[outs[-1]].argmax(1), y_t,
                       cost, cum[-1] if cfg else 0.0)
    print(f"\nEarly exiting, INT8 ({len(y_t)} test images):")
    for r in rows:
        print(f"  {r['profile']:<12s} acc {r['acc']:6.2f}%  exits {[round(s, 1) for s in r['exit_share']]}%"
              + (f"  cost {100 * r['cost'] / cum[-1]:5.1f}% of the backbone" if cfg else ""))
    if cfg:
        b = DP.best_ee(rows)
        print(f"Best-EE profile: {b['profile']} ({b['acc']:.2f}%, {100 * b['cost'] / cum[-1]:.1f}% of the backbone cost)")

## 6 · Compile to C source

In [ ]:
import time

model_file_path = tflite_to_compile_path

print(f"Loading TFLite model from: {model_file_path}")
with open(model_file_path, 'rb') as f:
    tfl_model_bytes = f.read()

try:
    import tflite
    tfl_model = tflite.Model.GetRootAsModel(tfl_model_bytes, 0)
except AttributeError:
    import tflite.Model
    tfl_model = tflite.Model.Model.GetRootAsModel(tfl_model_bytes, 0)

mod, params = tvm.relay.frontend.from_tflite(tfl_model)
target   = tvm.target.Target("c")
runtime  = Runtime("crt")
executor = Executor("aot", {"unpacked-api": True, "interface-api": "c",
                            "workspace-byte-alignment": 8})

if params:
    mod["main"] = tvm.relay.build_module.bind_params_by_name(mod["main"], params)

# ── Per-model compilation strategy ───────────────────────────
# Most models use the custom 3-phase pipeline (fold-before-QNN)
# at opt_level 3. Architectures like NASNet need the
# standard pipeline at a lower opt_level — see model registry.
USE_CUSTOM   = MODEL_CFG.get("tvm_custom_pipeline", True)
OPT_LEVEL    = MODEL_CFG.get("tvm_opt_level", TVM_OPT_LEVEL)

t0 = time.time()

if USE_CUSTOM:
    print(f"Compiling with custom pipeline, opt_level={OPT_LEVEL} ...")
    disabled_qnn_passes = ["qnn.Legalize", "qnn.Canonicalize"]
    if not QUANTIZE_FOR_TVM:
        print("  Disabling QNN-specific passes for unquantized model.")

    with tvm.transform.PassContext(
            opt_level=OPT_LEVEL,
            disabled_pass=disabled_qnn_passes if not QUANTIZE_FOR_TVM else [],
            config={"tir.disable_vectorize": True}):
        pre_passes = tvm.transform.Sequential([
            relay.transform.InferType(),
            relay.transform.FoldConstant(),
            relay.transform.SimplifyInference(),
        ])
        mod = pre_passes(mod)

        if QUANTIZE_FOR_TVM:
            qnn_passes = tvm.transform.Sequential([
                relay.qnn.transform.Legalize(),
                relay.qnn.transform.CanonicalizeOps(),
            ])
            mod = qnn_passes(mod)

        compiled = tvm.relay.build(mod, target, runtime=runtime,
                                   executor=executor, params={})
else:
    print(f"Compiling with standard pipeline, opt_level={OPT_LEVEL} ...")
    disabled_passes = ["relay.AlterOpLayout"]
    if not QUANTIZE_FOR_TVM:
        print("  Disabling QNN-specific passes for unquantized model.")
        disabled_passes.extend(["qnn.Legalize", "qnn.Canonicalize"])

    with tvm.transform.PassContext(
            opt_level=OPT_LEVEL,
            disabled_pass=disabled_passes,
            config={"tir.disable_vectorize": True}):
        compiled = tvm.relay.build(mod, target, runtime=runtime,
                                   executor=executor, params={})

elapsed = time.time() - t0
print(f"Compilation finished in {elapsed:.1f}s")
param_size = len(tvm.runtime.save_param_dict(params)) if params else len(tvm.runtime.save_param_dict(compiled.get_params()))
print(f"Params: {param_size:,} bytes")
tar_suffix = "_int8_aot.tar" if QUANTIZE_FOR_TVM else "_float32_aot.tar"
tar = OUTPUT_DIR / f"{MODEL_CFG['name']}_EE{tar_suffix}"
export_model_library_format(compiled, tar)
print(f"MLF archive → {tar}")
